# 02 — Forward models and surrogate libraries

Both physics arms are reduced to the same dimensionless surrogate: log-amplitude on a **resonance-aligned** axis u = f/f_res, indexed by contact stiffness and damping. One fitting loop then drives either model (`physrec.use_library`).

Build order: `libraries/buildlib.py` (EB, ~5 min) → `libraries/run_fem_ladder.py` + `build_fem_library.py` (FEM). This notebook assumes the libraries exist in `results/`.

In [ ]:
import sys; sys.path.insert(0,'..'); sys.path.insert(0,'../src')
from config import OUT
import physrec as P
print(P.use_library(str(OUT/'eblib.npz')))

### What is fitted

Four numbers per reveal step: log k₁ (bounded by the library), f_res (±2 % of the revealed peak), log damping, plus a closed-form gain. Objective is MSE in T(A)=log(A+F) over a high-SNR window (~32 of 412 bins) — fitting the full band drives k₁ to the library edge because 380 off-resonance bins outvote the resonance while the models sit ~8 dB above the measured floor there.

### Why resonance alignment

Interpolating the raw library across k₁ mixes peaks at different frequencies: 3.8 % rms / 42 % worst-case error. Aligned, the same library interpolates to ~0.1 %.

In [ ]:
import numpy as np
from run_phys import prep, truth_dns, score
x,f,A=prep(); TD,ir=truth_dns(x,f,A)
sel=sorted(np.linspace(0,len(x)-1,8).round().astype(int).tolist())
th=P.fit_eb(sel,A[sel],f)
print({k:round(v,3) if isinstance(v,float) else v for k,v in th.items() if k in ('k1','g','rms','f_res')})

### The k₁ identifiability problem

f₁ moves only +3.8 % over k₁ = 328→1291 N/m while f₂/f₁ moves +28 % — the mode ratio is ~7× more informative about k₁. `results/eb_f2f1.npy` holds the EB curve; the FEM ratios come from `eigen/frequencies_hz` in the ladder exports. Inverting the measured 3.080 through EB gives k₁ ≈ 1046 N/m (Hertz predicts 1310), 3× stiffer than the band-A fit.

**Open item:** extend both libraries past mode 2 (U_HI ≈ 3.5; EB FLIB to ~1.35 MHz) and fit both bands jointly with per-band SNR windows.

In [ ]:
r=np.load(str(OUT/'eb_f2f1.npy')); ok=np.isfinite(r[:,2])
k,rat=r[ok,0],r[ok,2]/r[ok,1]
import matplotlib.pyplot as plt
plt.semilogx(k,rat); plt.axhline(3.080,color='m',ls='--',label='measured')
plt.xlabel('k1 (N/m)'); plt.ylabel('f2/f1'); plt.legend();

### Model variants (see docs/fitting-and-model-content.md)

| variant | lateral spring | tip | reaches f₂/f₁=3.080? |
|---|---|---|---|
| EB k₂=0 (`eblib.npz`) | none | rigid | yes, at k₁≈1046 |
| EB k₂=k₁ rigid cone | k₁ | rigid | no — saturates 2.78 |
| EB k₂=k₁ + cone 100 N/m (`eblib_ic.npz`) | k₁ | compliant | **yes, at k₁≈1310 (Hertz!)** |
| FEM frictionless | none | elastic column | no — saturates 2.66 |
| FEM isotropic (`femlib_isotropic.npz`) | k₁ | elastic column | no — 2.94 at k₁=5000 |

`libraries/run_fem_etip.py` tests whether the FEM tip column explains its saturation.